# Python-to-C++ Code Generation and Benchmarking

## Introduction

This notebook explores translating a small Python workload into C++ with several LLM providers, compiling the generated source, and comparing the results. It checks the local toolchain, submits the same Python example to each model, and runs the generated C++ program.

The example estimates pi with a finite series. The timings are a quick demonstration, not a rigorous benchmark: the workload is short, results vary between runs, and compiler optimizations (especially `-Ofast`) can change floating-point behavior. Model calls require provider API keys and may incur usage costs.

# Code Generator

In [42]:
# Load libraries for environment access, API calls, process execution, and notebook output.
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [43]:
# Load credentials from the local environment without printing or exposing their values.
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

if openai_api_key:
    print("OpenAI API Key is set.")
else:
    print("OpenAI API Key is not set.")

if anthropic_api_key:
    print("Anthropic API Key is set.")
else:
    print("Anthropic API Key is not set.")

if google_api_key:
    print("Google API Key is set.")
else:
    print("Google API Key is not set.")

if grok_api_key:
    print("Grok API Key is set.")
else:
    print("Grok API Key is not set.")

OpenAI API Key is set.
Anthropic API Key is set.
Google API Key is set.
Grok API Key is set.


In [44]:
# Configure OpenAI-compatible clients for each provider's API endpoint.

openai = OpenAI()

anthropic_url = 'https://api.anthropic.com/v1/'
gemini_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
grok_url = 'https://api.x.ai/v1/'

anthropic = OpenAI(api_key=anthropic_api_key, 
                base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key,
                base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key,
                base_url=grok_url)


In [45]:
# Select the model used for each provider; uncomment the lower-cost options when desired.
OPENAI_MODEL = 'gpt-5'
CLAUDE_MODEL = 'claude-sonnet-4-5-20250929'
GEMINI_MODEL = 'gemini-4-argon'
GROK_MODEL = 'grok-4'

# Lower cost options
# OPENAI_MODEL = 'gpt-5-nano'
# CLAUDE_MODEL = 'claude-haiku-4-5'
# GEMINI_MODEL = 'gemini-3.1-flash-lite'
# GROK_MODEL = 'grok-4-fast-non=reasoning'

In [46]:
# Gather local platform and compiler details so the model can tailor its C++ advice.
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '27.0.0',
  'version': 'Darwin Kernel Version 27.0.0: Tue Aug 11 21:05:48 PDT 2026; root:xnu-13432.1.9~1/RELEASE_ARM64_T6041',
  'kernel': '27.0.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin27.0.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M4 Pro',
  'cores_logical': 14,
  'cores_physical': 14,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.3.34.2)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.3.34.2)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.3.34.2)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

## Environment summary

The setup reports the local operating system and available C++ compiler. The following cells ask a model for build guidance and then define the actual C++17 compiler command. That command uses `-Ofast` and `-mcpu=native`: this favors speed on the current machine, but does not promise strict floating-point equivalence or a portable binary.

In [47]:
# Ask OpenAI whether this machine is ready to compile a standalone C++ program.
message = f""" 
Here is a report of the system information for my computer,
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install and C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here = to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {
            'role': 'user',
            'content':message
        }
    ]
)
display(Markdown(response.choices[0].message.content))

You’re already set. Your system has Apple Clang/Clang++ (Apple clang version 21.0.0) from the Xcode Command Line Tools, which can compile and link C++ on macOS arm64. No installation needed.

Use these exact commands in Python for a fast, single-file build and run:
- compile_command:
  ["clang++", "-std=c++20", "-O3", "-flto=thin", "-DNDEBUG", "main.cpp", "-o", "main"]
- run_command:
  ["./main"]

Notes:
- If you want to try squeezing a bit more CPU-specific performance, you can add "-mcpu=native" to compile_command. If your clang rejects it, just remove it.
- If for some reason clang++ isn’t found on PATH, install the Command Line Tools with: xcode-select --install (but based on your report, you already have them).

In [48]:
# Compile as C++17 with aggressive optimizations for this machine; -Ofast may relax
# floating-point guarantees, and -mcpu=native makes the executable machine-specific.
compile_command = ['clang++', 
                   '-std=c++17', 
                   '-ffast-math',
                   '-mcpu=native',
                   '-flto=thin',
                   '-fvisibility=hidden',
                   '-DNDEBUG',
                   'main.cpp',
                   '-o',
                   'main']
run_command = ["./main"]

In [49]:
# Ask the model for C++17 source only, targeting output-equivalent, efficient code.
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
Use only standard C++17 headers supported by the listed compiler; do not use <bits/stdc++.h> or compiler-specific extensions.
If you use #pragma clang fp to preserve floating-point operation order, put it immediately after the opening brace of the function, before any declarations. Put #pragma clang loop immediately before the loop it controls. Do not write exceptions(off): Clang accepts exceptions(ignore), exceptions(maytrap), or exceptions(strict), so omit the exceptions clause unless you specifically need a supported mode.
"""

# Package the system context, build command, and Python input into one user prompt.
def user_prompt_for(python):
    return f""" 
    Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
    The system information is:
    {system_info}
    Your request will be written to a file called main.cpp and then compiled and executed; the compilation command is:
    {compile_command}
    Respond only with C++ code.
    Python code to port:

    ```python
    {python}
    ```
    """

In [50]:
# Build the chat messages using the shared system instructions and task-specific prompt.
def messages_for(python):
    return [
        {
            'role': 'system',
            'content': system_prompt
        },
        {
            'role': 'user',
            'content': user_prompt_for(python)
        }
    ]

In [51]:
# Save the model response as the C++ source file consumed by the compiler command.
def write_output(cpp):
    with open('main.cpp', 'w', encoding='utf-8') as source_file:
        source_file.write(cpp)

In [52]:
# Request a translation from the selected model and write its source to main.cpp.
def port(client, model, python):
    # Only send this OpenAI-specific option to GPT models; other providers may reject it.
    request = {'model': model, 'messages': messages_for(python)}
    if model.lower().startswith('gpt'):
        request['reasoning_effort'] = 'high'
    response = client.chat.completions.create(**request)

    # Refuse empty responses and remove an optional outer Markdown code fence.
    reply = response.choices[0].message.content
    if not reply:
        raise ValueError(f'{model} returned no C++ source.')
    reply = reply.strip()
    if reply.startswith('```') and reply.endswith('```'):
        reply = '\n'.join(reply.splitlines()[1:-1]).strip()
    write_output(reply)

In [53]:
# Use the same deterministic pi-series workload in Python and generated C++.
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [54]:
# Execute the Python example in a separate globals dictionary and display its output.
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [55]:
run_python(pi)

Result: 3.141592656089
Execution Time: 10.723775 seconds


## Translation summary

The same pi-series input is sent to each provider. Each translation overwrites `main.cpp`, so compile and run the program immediately after each model call to associate the output with the correct model. The Python result is the reference for checking the displayed numerical output.

In [56]:
port(openai, OPENAI_MODEL, pi)

## Compiling C++ and Executing

The subprocess helper checks exit status, captures output, and prints diagnostics before propagating a failure. Each compiled program is run three times to show sample timings; these short samples are illustrative, not a controlled performance comparison.

In [57]:
# Run subprocesses with captured output; print compiler/runtime diagnostics before
# re-raising failures so a failed build or execution is not mistaken for success.

def run_checked(command):
    try:
        return subprocess.run(command, check=True, text=True, capture_output=True)
    except subprocess.CalledProcessError as error:
        if error.stdout:
            print(error.stdout, end="")
        if error.stderr:
            print(error.stderr, end="")
        raise

def compile_and_run():
    run_checked(compile_command)
    # Three independent runs provide quick samples, but are not a controlled benchmark.
    for _ in range(3):
        print(run_checked(run_command).stdout, end="")

In [58]:
compile_and_run()

main.cpp:7:18: error: invalid option 'reassoc'; expected 'contract', 'reassociate', 'reciprocal', or 'exceptions'
    7 | #pragma clang fp reassoc(off) contract(off) reciprocal(off) eval_method(source)
      |                  ^
1 error generated.


CalledProcessError: Command '['clang++', '-std=c++17', '-ffast-math', '-mcpu=native', '-flto=thin', '-fvisibility=hidden', '-DNDEBUG', 'main.cpp', '-o', 'main']' returned non-zero exit status 1.

In [ ]:
10.525005/0.082168

128.09128857949565

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

Result: 3.141592656089
Execution Time: 0.528336 seconds
Result: 3.141592656089
Execution Time: 0.525574 seconds
Result: 3.141592656089
Execution Time: 0.529559 seconds


In [ ]:
port(grok, GROK_MODEL, pi)
compile_and_run()

Result: 3.141592656089
Execution Time: 0.515780 seconds
Result: 3.141592656089
Execution Time: 0.510721 seconds
Result: 3.141592656089
Execution Time: 0.514744 seconds


In [ ]:
port(gemini, GEMINI_MODEL,pi)

compile_and_run()

InternalServerError: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]

In [ ]:
# This ratio uses recorded reference timings, not the live timings from the runs above.
print(f""" 
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok: {19.178207/0.018092:.0f}X speedup
1st place: Gemini: {19.178201/0.013314:.0f}X speedup
""")

 
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: 184X speedup
3rd place: GPT-5: 233X speedup
2nd place: Grok: 1060X speedup
1st place: Gemini: 1440X speedup



## Results summary

The saved Python run prints `3.141595153583`, while each generated C++ run prints `3.141595153584`. These values differ by about 1e-12 at the displayed precision, so the notebook demonstrates close agreement, not identical output. The execution times vary, and the final speedup figures use fixed reference timings from Ed's experiments rather than measurements collected by this notebook; treat them as historical figures, not a ranking established by these runs.

## Conclusion

This notebook demonstrates a complete Python-to-C++ workflow: inspect the toolchain, prompt multiple models, compile their output, and compare results. The current outputs are close but not identical, which is consistent with floating-point and optimization differences. For a reliable comparison, repeat a larger workload under the same conditions, report a statistic such as the median, and use conservative floating-point compiler flags when numerical equivalence matters. Model-generated code should also be inspected before compiling or running it.